In [1]:
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback

from utils.sabre_env_wog_wc import SabreSwapEnv
from collections import deque

import cProfile
import pstats


In [2]:
class CurriculumCallback(BaseCallback):
    """Optimized callback for curriculum learning."""

    def __init__(self, max_level: int = 1000, success_threshold: float = 0.8, verbose: int = 0, max_window: int = 100):
        super().__init__(verbose)
        self.max_level = max_level
        self.max_window = max_window
        self.success_threshold = success_threshold
        
        self.level = 1
        self.episode_results = deque(maxlen=max_window)  # 자동 크기 제한
        self.success_count = 0
        
        # 로깅 주기 설정 (매 스텝이 아닌 주기적으로)
        self.log_freq = 100
        self.step_count = 0

    def _on_step(self) -> bool:
        dones = self.locals.get("dones", [])
        
        # 에피소드 완료된 환경들만 처리
        if any(dones):
            successed = self.training_env.env_method("is_success")
            
            for i, done in enumerate(dones):
                if done:
                    if len(self.episode_results) == self.max_window:
                        # 가장 오래된 결과 제거 시 success_count 업데이트
                        if self.episode_results[0]:
                            self.success_count -= 1
                    
                    # 새 결과 추가
                    success = successed[i]
                    self.episode_results.append(success)
                    if success:
                        print(f"Episode {len(self.episode_results)}: Success")
                        self.success_count += 1
                        print(f"Current success count: {self.success_count}")
            
            # 레벨 업 체크
            if len(self.episode_results) >= self.max_window:
                success_rate = self.success_count / len(self.episode_results)
                if success_rate >= self.success_threshold:
                    self.level = min(self.level + 1, self.max_level)
                    self.training_env.env_method("set_level", level=self.level)
                    
                    if self.verbose > 0:
                        print(f"Level increased to {self.level} (success rate: {success_rate:.3f})")
        
        # 주기적으로만 상세 로깅
        self.step_count += 1
        if self.step_count % self.log_freq == 0:
            episode_count = len(self.episode_results)
            success_rate = self.success_count / max(1, episode_count)
            
            self.logger.record("success_rate", success_rate)
            self.logger.record("level", self.level)
            
            # 환경 상태는 덜 자주 로깅
            if self.step_count % (self.log_freq * 5) == 0:
                front_layer_len = self.training_env.env_method("front_layer_size")
                swap_candidate_len = self.training_env.env_method("swap_candidate_size")
                reset_failed = self.training_env.env_method("get_reset_failed")
                
                self.logger.record("front_layer_size/mean", np.mean(front_layer_len))
                self.logger.record("swap_candidate_size/mean", np.mean(swap_candidate_len))
                self.logger.record("reset_failed/mean", np.mean(reset_failed))
        
        return True


In [3]:
def profile_training():
    env = make_vec_env(SabreSwapEnv, n_envs=8,
                       env_kwargs={"S_a": 10, "H": 10, "start_level": 3})
    model = PPO("MlpPolicy", env, verbose=1, device="cuda")
    model.learn(total_timesteps=10000, callback=CurriculumCallback())
    env.close()

# 프로파일링 실행
cProfile.run('profile_training()', 'profile_stats')
stats = pstats.Stats('profile_stats')
stats.sort_stats('tottime').print_stats(20)


Using cuda device


d:\lab\circuit_route\.venv\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Episode 8: Success
Current success count: 1
Episode 12: Success
Current success count: 2
Episode 16: Success
Current success count: 3
Episode 18: Success
Current success count: 4
All gates are executable.
Episode 19: Success
Current success count: 5
All gates are executable.
All gates are executable.
Episode 25: Success
Current success count: 6
All gates are executable.
All gates are executable.
Episode 43: Success
Current success count: 7
Episode 46: Success
Current success count: 8
All gates are executable.
Episode 61: Success
Current success count: 9
Episode 67: Success
Current success count: 10
Episode 68: Success
Current success count: 11
Episode 70: Success
Current success count: 12
Episode 71: Success
Current success count: 13
All gates are executable.
Episode 83: Success
Current success count: 14
Episode 84: Success
Current success count: 15
Episode 89: Success
Current success count: 16
Episode 90: Success
Current success count: 17
Episode 97: Success
Current success count: 18


KeyboardInterrupt: 